# CodeBridge AI: does the Python translation print the same thing as the C++ program?

**What our project does.** We asked Claude to translate 10 small C++ programs into Python. We tried three different prompts: V0, V1 and V2. A translated program can run without errors and still print something different, so we check every translation by comparing what it prints with what the real C++ program prints.

**What this notebook does.** It does not call any AI. It loads the translations that Claude already gave us and checks them, one step at a time, so you can see what happens.

**How to use it.** Click **Runtime**, then **Run all**. It takes about a minute. You do not need to log in or add any key. Under each step you will see a short explanation and then the result.

In [1]:
#@title Step 0: Load the project files (nothing to read here, just run it)
import base64, io, zipfile, pathlib
ZIP_B64 = """
...
AACmAKYARzQAAOPWAAAAAA==
"""
root = pathlib.Path("CodeBridgeAI")
zipfile.ZipFile(io.BytesIO(base64.b64decode(ZIP_B64.replace("\n", "")))).extractall(root)
tests = sorted(p.name for p in (root / "test-data" / "tests").iterdir() if p.is_dir())
replies = sorted(p.stem for p in (root / "test-data" / "replies").glob("*.txt"))
print("Project files loaded.")
print("C++ test programs:", ", ".join(tests))
print("Claude's saved translations for the prompts:", ", ".join(replies))

Project files loaded.
C++ test programs: T01, T02, T03, T04, T05, T06, T07, T08, T09, T10
Claude's saved translations for the prompts: V0, V1, V2


## Step 1: See one test from start to finish (T07)

T07 is a C++ program about unsigned numbers. Below you see what the real C++ program prints, then what Claude's Python translation prints for the plain prompt (V0) and for the better prompt (V1).

**What to look for:** a line marked `<<< different from C++` means the translation printed something else. With V0, one line prints `True` where C++ prints `1`. With V1, nothing is different.

In [2]:
import pathlib, subprocess, sys
root = pathlib.Path("CodeBridgeAI/test-data/tests/T07")
expected = (root / "expected_output.txt").read_text().strip().split("\n")

def python_output(path):
    path = path.resolve()
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=path.parent)
    return (r.stdout if r.returncode == 0 else r.stderr).strip().split("\n")

print("WHAT THE REAL C++ PROGRAM PRINTS (test T07):")
for line in expected:
    print("    " + line)

for version, name in [("V0", "plain prompt"), ("V1", "better prompt")]:
    print()
    print(f"WHAT THE {version} PYTHON TRANSLATION PRINTS ({name}):")
    got = python_output(root / version / "code.py")
    for i, line in enumerate(got):
        same = i < len(expected) and line == expected[i]
        print("    " + line + ("" if same else "      <<< different from C++"))
    print("    Result:", (root / version / "result.txt").read_text().strip())

WHAT THE REAL C++ PROGRAM PRINTS (test T07):
    0u - 1 = 4294967295
    250 + 10 as uint8_t = 4
    uint8_t 65 printed directly: A
    uint8_t 65 printed with +: 65
    4000000000 + 500000000 as uint32_t = 205032704
    uint16_t 65535 + 1 = 0
    uppercase: A
    3000000000u * 2 = 1705032704
    5u - 10u = 4294967291
    (5u - 10u) > 0 is 1

WHAT THE V0 PYTHON TRANSLATION PRINTS (plain prompt):
    0u - 1 = 4294967295
    250 + 10 as uint8_t = 4
    uint8_t 65 printed directly: A
    uint8_t 65 printed with +: 65
    4000000000 + 500000000 as uint32_t = 205032704
    uint16_t 65535 + 1 = 0
    uppercase: A
    3000000000u * 2 = 1705032704
    5u - 10u = 4294967291
    (5u - 10u) > 0 is True      <<< different from C++
    Result: WRONG

WHAT THE V1 PYTHON TRANSLATION PRINTS (better prompt):
    0u - 1 = 4294967295
    250 + 10 as uint8_t = 4
    uint8_t 65 printed directly: A
    uint8_t 65 printed with +: 65
    4000000000 + 500000000 as uint32_t = 205032704
    uint16_t 65535 + 1 = 

## Step 2: Check that the correct answers really come from C++

Every test has a saved "expected output". It was made by running the real C++ program. To be sure, this step compiles all 10 C++ programs again with a C++ compiler (g++) and checks that they print exactly the saved text.

**What to look for:** ten lines saying OK, then "10 of 10 expected outputs confirmed".

In [3]:
import pathlib, subprocess
root = pathlib.Path("CodeBridgeAI/test-data/tests")
confirmed = 0
for t in sorted(p for p in root.iterdir() if p.is_dir()):
    exe = t / "prog"
    compiled = subprocess.run(["g++", "-std=c++17", "-o", str(exe), str(t / "program.cpp")], capture_output=True, text=True)
    if compiled.returncode != 0:
        print(t.name + ": the C++ program did not compile")
        continue
    printed = subprocess.run([str(exe)], capture_output=True, text=True, timeout=20).stdout
    same = printed.strip() == (t / "expected_output.txt").read_text().strip()
    confirmed += same
    print(t.name + ": real C++ output matches the saved expected output ->", "OK" if same else "DIFFERENT")
print()
print(confirmed, "of 10 expected outputs confirmed with a real C++ compiler.")

T01: real C++ output matches the saved expected output -> OK
T02: real C++ output matches the saved expected output -> OK
T03: real C++ output matches the saved expected output -> OK
T04: real C++ output matches the saved expected output -> OK
T05: real C++ output matches the saved expected output -> OK
T06: real C++ output matches the saved expected output -> OK
T07: real C++ output matches the saved expected output -> OK
T08: real C++ output matches the saved expected output -> OK
T09: real C++ output matches the saved expected output -> OK
T10: real C++ output matches the saved expected output -> OK

10 of 10 expected outputs confirmed with a real C++ compiler.


## Step 3: Run all the translations and score them

Now our script (`code/run.py`) runs every Python translation: 3 versions times 10 tests. It compares each result with the expected output.

- **PASS** means the output is exactly the same as the C++ output.
- **WRONG** means the program ran but printed something different.
- **ERROR** means the program crashed.

**What to look for:** the table, and at the bottom the score of each version.

In [4]:
import subprocess, sys, csv, pathlib
r = subprocess.run([sys.executable, "CodeBridgeAI/code/run.py"], capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print(r.stderr)
print("-" * 60)
names = {"V0": "plain prompt", "V1": "better prompt", "V2": "better prompt + repair"}
rows = list(csv.reader(open("CodeBridgeAI/test-data/results/results.csv")))
for row in rows:
    if len(row) == 4 and row[0] in names:
        print(f"{row[0]} ({names[row[0]]}): {row[1]} of {row[2]} tests correct ({row[3]}%)")


test  V0        V1        V2        
T01   PASS      PASS      PASS      
T02   PASS      PASS      PASS      
T03   WRONG     PASS      PASS      
T04   PASS      PASS      PASS      
T05   WRONG     ERROR     PASS      
T06   WRONG     PASS      PASS      
T07   WRONG     PASS      PASS      
T08   WRONG     PASS      PASS      
T09   WRONG     PASS      PASS      
T10   PASS      PASS      PASS      

V0: 4/10 passed (40%)  ########
V1: 9/10 passed (90%)  ##################
V2: 10/10 passed (100%)  ####################

>>> NEXT STEP: Everything is done. See test-data/results/.

------------------------------------------------------------
V0 (plain prompt): 4 of 10 tests correct (40%)
V1 (better prompt): 9 of 10 tests correct (90%)
V2 (better prompt + repair): 10 of 10 tests correct (100%)


## Step 4: Why did some tests fail?

This is the list of failures. Each line gives the test, the version, and the first line where the output was different. Most V0 failures are small printing differences, for example extra spaces, `True` instead of `1`, or `7.0` instead of `7`.

In [5]:
import pathlib
text = (pathlib.Path("CodeBridgeAI/test-data/results/failures.txt")).read_text()
print(text)

FAILURES (auto-generated: first place where the Python output differs from the real C++ output)

T03 V0 WRONG: line 8:  expected [Size: 5, front: 0, back: 16]   got [Size: 5 , front: 0 , back: 16]
T05 V0 WRONG: line 5:  expected [7]   got [7.0]
T05 V1 ERROR: error: TypeError: unsupported operand type(s) for -: 'str' and 'str'
T06 V0 WRONG: line 1:  expected [7 / 2 = 3, 7 % 2 = 1]   got [7 / 2 = 3 , 7 % 2 = 1]
T07 V0 WRONG: line 10:  expected [(5u - 10u) > 0 is 1]   got [(5u - 10u) > 0 is True]
T08 V0 WRONG: line 1:  expected [0.333333]   got [0.3333333333333333]
T09 V0 WRONG: line 9:  expected [b1.v = 1, b2.v = 2]   got [b1.v = 1 , b2.v = 2]


## Step 5: The one crash (T05) and how the repair prompt fixed it

V1 crashed on T05. For V2 we showed Claude the failed test, the expected output and the real error. We asked what went wrong and for a fixed version.

**What to look for:** the error from V1, Claude's explanation, and then PASS for V2.

In [6]:
import pathlib
t = pathlib.Path("CodeBridgeAI/test-data/tests/T05")
print("V1 result for T05:", (t / "V1" / "result.txt").read_text().strip())
print("The error message was:")
print("    " + (t / "V1" / "actual_output.txt").read_text().strip().split("\n")[-1])
print()
print("What Claude said after we showed it the failed test (V2):")
print((t / "V2" / "cause_of_fix.txt").read_text().strip())
print()
print("V2 result for T05:", (t / "V2" / "result.txt").read_text().strip())
print("What the fixed Python code prints:")
print((t / "V2" / "actual_output.txt").read_text())

V1 result for T05: ERROR
The error message was:
    TypeError: unsupported operand type(s) for -: 'str' and 'str'

What Claude said after we showed it the failed test (V2):
Cause: The initialization `values[0] - values[0]` works for numeric values but fails for strings because Python does not support string subtraction. C++ initializes `T total = T()`, which creates the default value for the type: `0` for numbers and an empty string for strings.

V2 result for T05: PASS
What the fixed Python code prints:
9
2.5
pear
10
7
abc
30 20 10 


## What this shows

- The plain prompt (V0) got 4 of 10 tests right.
- The better prompt (V1) got 9 of 10 right.
- After the repair step (V2), all 10 were right.

This is a small test: 10 programs and one run for each prompt. Passing means the output is identical. It does not prove that every translation is a perfect rewrite.

**Want to try it?** Open the folder icon on the left, go to `CodeBridgeAI/test-data/replies/`, and change a reply file. Then run Step 3 again to see the new score.